## Test 1 : Humaneval testing using Qwen/Qwen2.5-Coder-3B

In [ ]:
## First clone the github repo of human-eval
# !git clone https://github.com/openai/human-eval
# !pip install -e human-eval

In [ ]:
# Install vLLM from pip:
# !pip install vllm

In [ ]:
# Load and run the model:
# !vllm serve "Qwen/Qwen2.5-Coder-3B"

In [ ]:
# !pip install -U transformers

In [ ]:
#Importing libraries
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from human_eval.data import read_problems
from human_eval.evaluation import evaluate_functional_correctness

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


I am an AI language model created by OpenAI. I am designed to assist and provide information to users. What would you like to know or discuss?<|file_sep|><|fim_prefix|>/README.md
# system



In [ ]:
model_name = "Qwen/Qwen2.5-Coder-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def generate_completion(prompt, max_new_tokens=256):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,      # deterministic for pass@1
    )

    # Only new tokens
    completion = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return completion


In [ ]:
problems = read_problems()   # loads all 164 tasks


In [ ]:
problems['HumanEval/0']['prompt']

'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n'

In [ ]:
samples = []

for task_id, problem in problems.items():
    prompt = problem["prompt"]  # includes function header + docstring
    completion = generate_completion(prompt)

    samples.append({
        "task_id": task_id,
        "completion": completion,
    })


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end ge

In [ ]:
    import json
    file_path = "samples.jsonl"
    with open(file_path, "w") as json_file:
        json.dump(samples, json_file, indent=4)

In [ ]:
from human_eval.evaluation import evaluate_functional_correctness

results = evaluate_functional_correctness(
    sample_file="samples.jsonl",
    k=[1, 10, 100],
)
print(results)


Reading samples...


0it [00:00, ?it/s]


JSONDecodeError: Expecting value: line 2 column 1 (char 2)

In [ ]:
with open("samples.jsonl", "w") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")

In [ ]:
results = evaluate_functional_correctness(
    sample_file="samples.jsonl",
    k=[1, 10, 100],
)
print(results)

Reading samples...


164it [00:00, 34191.56it/s]


Running test suites...


100%|██████████| 164/164 [00:13<00:00, 12.37it/s]


Writing results to samples.jsonl_results.jsonl...


100%|██████████| 164/164 [00:00<00:00, 41397.80it/s]

{'pass@1': np.float64(0.09146341463414634)}


## Test 2 : Humaneval testing using Qwen/Qwen2.5-Coder-3B-Instruct

In [ ]:

# Qwen/Qwen2.5-Coder-3B-Instruct


In [ ]:
#Importing libraries
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from human_eval.data import read_problems
from human_eval.evaluation import evaluate_functional_correctness

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 5438161a-f266-44a5-86a0-57fe7b8bcb3b)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-3B-Instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


tokenizer_config.json: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7c4f02ff-1e5b-4285-9cd0-1bdd4762e29f)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-3B-Instruct/resolve/main/vocab.json
Retrying in 1s [Retry 1/5].


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

I am Qwen, an AI language model developed by Alibaba Cloud. I'm designed to assist with answering questions, providing information, and engaging in conversation on a wide range of topics. How can I


In [ ]:
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def generate_completion(prompt, max_new_tokens=256):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.001,      # deterministic for pass@1
    )

    # Only new tokens
    completion = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return completion


In [ ]:
problems = read_problems()   # loads all 164 tasks


In [ ]:
samples1 = []

for task_id, problem in problems.items():
    prompt = problem["prompt"]  # includes function header + docstring
    completion = generate_completion(prompt)

    samples1.append({
        "task_id": task_id,
        "completion": completion,
    })


In [ ]:
    import json
    file_path = "samples.jsonl"
    with open(file_path, "w") as json_file:
        json.dump(samples, json_file, indent=4)

In [ ]:
with open("samples-instruct-model-qwen-results.jsonl", "w") as f:
    for s in samples1:
        f.write(json.dumps(s) + "\n")

In [ ]:
results = evaluate_functional_correctness(
    sample_file="samples.jsonl",
    k=[1, 10, 100],
)
print(results)

## Test 3: Humaneval testing using Qwen/Qwen2.5-Coder-3B with prompt engg techniques

### Chain of thought